# Ocean colour - Sentinel-3 OLCI chlorophyll monthly via openEO

The `sentinel-3-olci-chlorophyll-monthly` recipe loads OLCI Level-2 ocean chlorophyll (neural-net retrieval, `CHL_NN`) and reduces it to a **monthly mean** server-side (NetCDF). A starting point for marine productivity and water-quality monitoring. We use a coastal box and one month.

```bash
pip install earthlens[openeo]
```

## Setup

Imports for the whole notebook in one place: `os` and `pathlib.Path` for the credential checks, and the unified `EarthLens` entry point from `earthlens`.

In [ ]:
import os
from pathlib import Path

from earthlens.core import EarthLens

## Credentials

openEO authenticates with CDSE via OIDC. This notebook runs the live download for real when credentials are present (env `OPENEO_CLIENT_ID` / `OPENEO_CLIENT_SECRET`, or a cached refresh token from a prior interactive login) and prints a skip note otherwise - so it executes cleanly with or without secrets. See the Authentication page.

### Detecting available credentials

`has_openeo_credentials()` reports whether any OIDC credential source is configured: a headless client id/secret pair, a refresh token, or a cached `~/.openeo` login from a prior interactive sign-in.

In [ ]:
def has_openeo_credentials() -> bool:
    """Whether some openEO OIDC credential source is available."""
    if os.environ.get("OPENEO_CLIENT_ID") and os.environ.get("OPENEO_CLIENT_SECRET"):
        return True
    if os.environ.get("OPENEO_REFRESH_TOKEN"):
        return True
    return (Path.home() / ".openeo").exists()

### A run-or-skip wrapper

`run_or_skip()` calls `facade.download(...)` only when credentials exist, otherwise it prints a skip note and returns an empty list. This keeps the notebook executing top-to-bottom with no errors whether or not CDSE OIDC credentials are configured, so the docs build never needs secrets.

In [ ]:
def run_or_skip(facade, **download_kwargs):
    """Run the live download when credentials exist, else print a skip note."""
    if not has_openeo_credentials():
        print(
            "No openEO OIDC credentials found - skipping the live download.\n"
            "Set OPENEO_CLIENT_ID / OPENEO_CLIENT_SECRET (headless) or sign in once\n"
            "interactively (see the Authentication page) to run this cell for real."
        )
        return []
    paths = facade.download(**download_kwargs)
    for path in paths:
        print(path)
    return paths

## Build the request

Construct the `EarthLens` facade for the openEO backend: the `sentinel-3-olci-chlorophyll-monthly` recipe over a coastal box (`aoi`) for July 2023, writing NetCDF to `data/openeo-chl`.

In [ ]:
facade = EarthLens(
    data_source='openeo',
    dataset='sentinel-3-olci-chlorophyll-monthly',
    variables=[],
    start='2023-07-01',
    end='2023-07-31',
    aoi=[-1.0, 36.0, 4.0, 40.0],
    path='data/openeo-chl',
)

### Inspect the resolved recipe

The facade resolves the recipe name to a concrete openEO collection and band list; printing them confirms what will be requested server-side.

In [ ]:
g = facade.datasource._resolved['sentinel-3-olci-chlorophyll-monthly']
print('collection:', g.collection_id, '| bands:', g.bands)

## Download (server-side execution)

Hand the facade to `run_or_skip()`: with credentials it executes the monthly-mean reduction on the openEO backend and returns the written NetCDF path(s); without them it prints the skip note and returns an empty list.

In [ ]:
paths = run_or_skip(facade)
paths